In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. データの読み込みと確認

初めに使用するライブラリを読み込みます。numpy, pandas, sklearnはよく使用するので、ブックマークしておくと良いと思います。<br>
[numpy](https://numpy.org/doc/1.21/index.html#)
[pandas](https://pandas.pydata.org/docs/#)
[matplotlib](https://matplotlib.org/stable/index.html)
[sklearn](https://scikit-learn.org/stable/index.html)

In [ ]:
# ライブラリの読み込み
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn import ensemble
from sklearn import linear_model
from sklearn import model_selection
from sklearn import preprocessing

# 不要な警告を無視する
import warnings
warnings.filterwarnings('ignore')

pandasのread_csv関数を用いて、分析する訓練データtrain.csvとテストデータtest.csvを読み込みます。<br>
[read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html?highlight=read_csv#pandas.read_csv)

In [ ]:
# 学習用データ、テストデータ、提出サンプルデータの読み込み
train = pd.read_csv('../input/titanic/train.csv')
test = pd.read_csv('../input/titanic/test.csv')
sample = pd.read_csv('../input/titanic/gender_submission.csv')

データを見ていく上で、まず初めにデータのサイズを確認してみます。

In [ ]:
# データサイズの確認
print('学習データのサイズ:', train.shape)
print('テストデータのサイズ:', test.shape)

データの情報を確認します。<br>
[info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html?highlight=info#pandas.DataFrame.info)

In [ ]:
# データの情報の確認
print(train.info(), '\n')
print(test.info())

学習データ、テストデータ、提出サンプルデータについて先頭の5行を表示します。

In [ ]:
# 学習データの先頭5行を表示
train.head()

In [ ]:
# テストデータの先頭5行を表示
test.head()

In [ ]:
# 提出サンプルデータの先頭5行を表示
sample.head()

学習用データを特徴量と目的変数に分けます。この際、特徴量には_x、目的変数には_yをつけました。目的変数は'Survived'です。テストデータには目的変数はなく特徴量だけなので分ける必要はないです。学習用データと同じようにテストデータには_xをつけました。<br>
[drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html?highlight=drop#pandas.DataFrame.drop)

In [ ]:
# 学習用データを特徴量と目的変数に分ける
train_x = train.drop(['Survived'], axis=1)
train_y = train['Survived']

# テストデータは特徴量のみなのでそのままで良い
test_x = test.copy()

In [ ]:
# 'Survived'の分布
sns.countplot(data=train, x='Survived')

# 2. 特徴量の作成

学習用データとテストデータの欠損値を確認します。train_x.isnull().sum()とすると、学習用データの特徴量ごとの欠損数を出力することができます。さらにtrain_x.isnull().sum().sort_values(ascending=False)を加えることで、欠損数が多い順に並び替えることができます。

In [ ]:
#学習データの欠損値を確認する
print('訓練データの欠損値:\n', train_x.isnull().sum().sort_values(ascending=False), '\n')
#テストデータの欠損値を確認する
print('テストデータの欠損値:\n', test_x.isnull().sum().sort_values(ascending=False))

'PassengerId'は乗客に番号を振っているだけであり、目的変数に影響を与えないため、削除します。

In [ ]:
train_x = train_x.drop(['PassengerId'], axis=1)
test_x = test_x.drop(['PassengerId'], axis=1)

'Name', 'Ticket', 'Cabin'も上手く使えば予測に有用ですが、煩雑な処理
が必要そうなので、今回はこれらの変数を使わないことにします。

In [ ]:
# 欠損値が多すぎる特徴量、予測に無関係と考えられる特徴量を削除する
drop_col = ['Name','Ticket', 'Cabin']
train_x = train_x.drop(drop_col, axis=1)
test_x = test_x.drop(drop_col, axis=1)

もう一度欠損数を確認します。

In [ ]:
#学習データの欠損値を確認する
print('訓練データの欠損値:\n', train_x.isnull().sum().sort_values(ascending=False), '\n')
#テストデータの欠損値を確認する
print('テストデータの欠損値:\n', test_x.isnull().sum().sort_values(ascending=False))

'Age', 'Embarked', 'Fare'は欠損値があるため補完します。'Age'は数値変数、'Embarked'はカテゴリ変数、'Fare'は数値変数です。数値変数であるAge, Fareは平均値、カテゴリ変数であるEmbarkedは最頻値で補完します。

In [ ]:
# 'Age'のヒストグラム
sns.histplot(data=train_x, x='Age')

In [ ]:
# Ageカラムの欠損値を平均値で補完する
train_x['Age'] = train_x['Age'].fillna(train_x['Age'].mean())
test_x['Age'] = test_x['Age'].fillna(test_x['Age'].mean())

In [ ]:
# 'Embarked'のカウントプロット
sns.countplot(data=train_x, x='Embarked')

In [ ]:
# Embarkedカラムの欠損値を最頻値で補完する
train_x['Embarked'] = train_x['Embarked'].fillna('S')
test_x['Embarked'] = test_x['Embarked'].fillna('S')

In [ ]:
# 'Fare'のヒストグラム
sns.histplot(data=train_x, x='Fare')

In [ ]:
# Fareカラムの欠損値を平均値で補完する
#print(train_x['Fare'].dtype)
train_x['Fare'] = train_x['Fare'].fillna(train_x['Fare'].mean())
test_x['Fare'] = test_x['Fare'].fillna(test_x['Fare'].mean())

欠損値を全て補完したことを確認します。

In [ ]:
# 欠損値を全て補完したことを確認する
print(train_x.isnull().sum(), '\n')
print(test_x.isnull().sum())

以上で欠損値の補完が完了しました。次はカテゴリ変数の変換をします。今回はラベルエンコーディングします。

In [ ]:
# データの情報の確認
print(train_x.info(), '\n')
print(test_x.info())

Dtypeがobjectとなっている変数がカテゴリ変数です。したがって、'Sex'と'Embarked'についてラベルエンコーディングする必要があります。まずは'Sex'についてラベルエンコーディングします。'Sex'に含まれる水準を確認します。

In [ ]:
print(train_x['Sex'].unique())
print(test_x['Sex'].unique())

'Sex'カラムにはmale, femaleが含まれることが分かりました。maleは0、femaleは1に変換したいと思います。

In [ ]:
# 'Sex'をマッピング　male:0, female:1
sex_mapping = {"male":0, "female":1}
train_x["Sex"] = train_x["Sex"].map(sex_mapping)
test_x["Sex"] = test_x["Sex"].map(sex_mapping)

次に'Embarked'カラムについてラベルエンコーディングします。'Embarked'に含まれる水準を確認します。

In [ ]:
print(train_x['Embarked'].unique())
print(test_x['Embarked'].unique())

'Embarked'カラムにはS, C, Qが含まれることがわかりました。Sは0、Cは1、Qは2に変換したいと思います。

In [ ]:
# Embarkedをマッピング　S:0, C:1, Q:2
embarked_mapping = {'S':0, 'C':1, 'Q':2}
train_x['Embarked'] = train_x['Embarked'].map(embarked_mapping)
test_x['Embarked'] = test_x['Embarked'].map(embarked_mapping)

再びデータの情報を確認します。

In [ ]:
# データの情報の確認
print(train_x.info(), '\n')
print(test_x.info())

全ての変数を数値変数に変換できたことが分かります。最後に数値変数を標準化します。

In [ ]:
# 標準化
scaler = preprocessing.StandardScaler()
scaler.fit(train_x)
train_x = scaler.transform(train_x)
test_x = scaler.transform(test_x)

# 3. モデリング

性能を評価するには、データの分割をする必要があります。今回は分類タスクなので、層化交差検証を用います。

In [ ]:
# 分割方法の指定
skf = model_selection.StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
# モデルの作成と評価(RidgeClassifier)
rc = linear_model.RidgeClassifier(random_state=0)
rc_results = model_selection.cross_validate(rc, train_x, train_y, scoring='accuracy', cv=skf)
rc_results['test_score'].mean()

In [ ]:
# モデルの作成と評価(RandomForestClassifier)
rfc = ensemble.RandomForestClassifier(random_state=0)
rfc_results = model_selection.cross_validate(rfc, train_x, train_y, scoring='accuracy', cv=skf)
rfc_results['test_score'].mean()

In [ ]:
# モデルの作成と評価 - チューニングあり(RandomForestClassifier)
# ハイパーパラメータ
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 2, 5],
}

tune_rfc = model_selection.GridSearchCV(rfc, param_grid=param_grid, scoring = 'accuracy', 
                                        cv = skf)
                                        
                                        
tune_rfc.fit(train_x, train_y)
print("最もよいパラメータ: ", tune_rfc.best_params_)
print("検証データの平均値: ", tune_rfc.cv_results_['mean_test_score'][tune_rfc.best_index_])

In [ ]:
tune_rfc.best_estimator_

# 4. 提出物の作成

In [ ]:
# 学習データ全体でモデルの学習をする
tune_rfc.best_estimator_.fit(train_x, train_y)

# テストデータに対して予測する
predict = tune_rfc.best_estimator_.predict(test_x)

In [ ]:
# 提出用ファイルの作成
submit = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': predict})
submit.to_csv('submit_1.csv', index=False)